# Relationship Matrix for Example 6

This notebook explains the matrix previously called the **relationship matrix** for **Example 6** in `rank1_qk_experiment.ipynb` and referenced in `SESSION_HANDOFF_2026-03-25.md`.

For this example, the definitions are:

$$
Q = X W_Q, \quad K = X W_K, \quad R = W_Q W_K^T, \quad S = X R X^T.
$$

The goals here are:

1. Write down the exact numerical value of $R$.
2. Explain what structure it has in Example 6.
3. Check whether the experiment is invariant to adding a constant to any one row of $R$.

The key subtlety is that there are two different questions:

- Is the **matrix $R$ itself** unchanged by this operation? No.
- Are the **attention probabilities and final outputs** unchanged? For this Example 6 setup, yes.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

np.set_printoptions(suppress=True, linewidth=160, precision=6)
torch.set_printoptions(precision=6, sci_mode=False, linewidth=160)

V = 5
L = 4
d = V + L
d_k = 1
d_v = 2
alpha6 = 6.0
gamma6 = 5.0

dim_labels = [f't{i}' for i in range(V)] + [f'p{i}' for i in range(L)]
validation_sequences = {
    'true_branch': [3, 0, 1, 0],
    'first_token_false': [0, 0, 1, 3],
    'pos2_false': [3, 0, 0, 1],
    'both_false': [1, 2, 0, 4],
}

def get_input_matrix(token_indices):
    tokens = torch.tensor(token_indices)
    content = F.one_hot(tokens, num_classes=V).float()
    position = F.one_hot(torch.arange(L), num_classes=L).float()
    return torch.cat([content, position], dim=-1)

In [ ]:
# Example 6 parameters from the original notebook
W_Q6 = torch.nn.Linear(d, d_k, bias=False)
W_K6 = torch.nn.Linear(d, d_k, bias=False)

with torch.no_grad():
    W_Q6.weight.zero_()
    W_Q6.weight[0, V + 3] = 1.0

    W_K6.weight.zero_()
    W_K6.weight[0, 3] = -4.0
    W_K6.weight[0, 1] = 4.0
    W_K6.weight[0, V + 0] = 4.0
    W_K6.weight[0, V + 1] = -6.0
    W_K6.weight[0, V + 2] = -1.0
    W_K6.weight[0, V + 3] = -6.0

W_V6 = torch.zeros(d, d_v)
W_V6[1, 0] = 1.0
W_V6[V + 0, 0] = -1.0
W_V6[:, 1] = 0.5

W_O6 = torch.zeros(d_v, V)
W_O6[0, 2] = gamma6
W_O6[0, 4] = -gamma6
W_O6[1, 4] = gamma6

W_Q_math = W_Q6.weight.T
W_K_math = W_K6.weight.T
R6 = W_Q_math @ W_K_math.T

print('W_Q (mathematical d x 1 column):')
print(W_Q_math.detach().numpy())
print('\nW_K (mathematical d x 1 column):')
print(W_K_math.detach().numpy())
print('\nR = W_Q W_K^T (shape d x d):')
print(R6.detach().numpy())

## Exact Numerical Value of $R$

Because $W_Q = e_{p3}$ in Example 6, the query column has a single 1 in the row for the final-position basis vector $p_3$. Therefore

$$
R = W_Q W_K^T = e_{p3} \, w_K^T,
$$

so **all rows of $R$ are zero except the row indexed by $p_3$**. Numerically, with row/column ordering

$$
[t_0, t_1, t_2, t_3, t_4, p_0, p_1, p_2, p_3],
$$

the matrix is

$$
R = \begin{bmatrix}
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 4 & 0 & -4 & 0 & 4 & -6 & -1 & -6
\end{bmatrix}.
$$

So the relationship matrix is **rank 1**, and it says: only the query feature $p_3$ matters, and when that query is active it scores key-space features according to the last row above.

In [ ]:
print('Labeled nonzero entries of R:')
for i in range(d):
    for j in range(d):
        value = R6[i, j].item()
        if abs(value) > 1e-12:
            print(f'  R[{i},{j}] = {value:+.1f}   ({dim_labels[i]} -> {dim_labels[j]})')

print('\nRow sums of R:')
for i in range(d):
    print(f'  row {i} ({dim_labels[i]}): {R6[i].sum().item():+.1f}')

## What Happens If We Add a Constant to One Row of $R$?

Pick a row index $r$ and a constant $c$. Adding $c$ to every entry of row $r$ means replacing $R$ by

$$
R' = R + c \, e_r \mathbf{1}^T,
$$

where $e_r$ is the $r$-th basis column and $\mathbf{1}$ is the all-ones column in feature space. The new score matrix is

$$
S' = X R' X^T = X R X^T + c \, X e_r \mathbf{1}^T X^T.
$$

Now in this toy embedding, every token-position vector has exactly two ones: one token basis vector and one position basis vector. Therefore for every context row $x_j$,

$$
\mathbf{1}^T x_j^T = 2.
$$

So

$$
\mathbf{1}^T X^T = [2,2,2,2],
$$

and hence

$$
S' - S = 2c \, (X e_r) \mathbf{1}_L^T.
$$

This means: for each fixed query row $i$, the perturbation adds the **same constant to every column $j$**. Row-wise softmax is invariant to this operation, so

$$
\mathrm{softmax}(S'_{i,:}) = \mathrm{softmax}(S_{i,:})
$$

for every row $i$. Therefore in Example 6:

- the raw relationship matrix $R$ changes,
- the raw score matrix $S$ changes,
- but the attention matrix $P$, attention output $H$, logits $z$, and probabilities $p_{out}$ stay the same.

So the correct statement is:

- **not invariant at the level of $R$ or $S$**,
- **invariant at the level of attention probabilities and final predictions**.

In [ ]:
def forward_from_R(seq, R):
    X = get_input_matrix(seq)
    S = X @ R @ X.T
    P = F.softmax(alpha6 * S, dim=-1)
    V_mat = X @ W_V6
    H = P @ V_mat
    h_last = H[-1]
    z = h_last.unsqueeze(0) @ W_O6
    p_out = F.softmax(z, dim=-1).squeeze()
    return {
        'X': X.detach().clone(),
        'S': S.detach().clone(),
        'P': P.detach().clone(),
        'H': H.detach().clone(),
        'z': z.detach().clone(),
        'p_out': p_out.detach().clone(),
    }

def add_constant_to_row(R, row_index, constant):
    R_new = R.clone()
    R_new[row_index, :] = R_new[row_index, :] + constant
    return R_new

row_constants = [-3.0, 2.5]
rows_to_test = list(range(d))

results = []
for seq_name, seq in validation_sequences.items():
    base = forward_from_R(seq, R6)
    for row_index in rows_to_test:
        for constant in row_constants:
            R_mod = add_constant_to_row(R6, row_index, constant)
            mod = forward_from_R(seq, R_mod)
            results.append({
                'sequence': seq_name,
                'row_index': row_index,
                'constant': constant,
                'max_abs_diff_S': (mod['S'] - base['S']).abs().max().item(),
                'max_abs_diff_P': (mod['P'] - base['P']).abs().max().item(),
                'max_abs_diff_H': (mod['H'] - base['H']).abs().max().item(),
                'max_abs_diff_z': (mod['z'] - base['z']).abs().max().item(),
                'max_abs_diff_p_out': (mod['p_out'] - base['p_out']).abs().max().item(),
            })

print(f'Tested {len(results)} perturbations total.')
print('For each validation sequence, each row of R was shifted by -3.0 and +2.5.')

In [ ]:
# Show a compact verification table
print('Verification summary:')
print('sequence               row  const   max|dS|      max|dP|      max|dH|      max|dz|      max|dp_out|')
print('-' * 108)
for item in results:
    print(
        f"{item['sequence']:<22} {item['row_index']:>3d}  {item['constant']:>5.1f}   "
        f"{item['max_abs_diff_S']:>10.6f}  {item['max_abs_diff_P']:>11.6e}  "
        f"{item['max_abs_diff_H']:>11.6e}  {item['max_abs_diff_z']:>11.6e}  "
        f"{item['max_abs_diff_p_out']:>13.6e}"
    )

max_dS = max(item['max_abs_diff_S'] for item in results)
max_dP = max(item['max_abs_diff_P'] for item in results)
max_dH = max(item['max_abs_diff_H'] for item in results)
max_dz = max(item['max_abs_diff_z'] for item in results)
max_dp = max(item['max_abs_diff_p_out'] for item in results)

print('\nWorst-case differences across all perturbations:')
print(f'  max|dS|      = {max_dS:.6f}')
print(f'  max|dP|      = {max_dP:.6e}')
print(f'  max|dH|      = {max_dH:.6e}')
print(f'  max|dz|      = {max_dz:.6e}')
print(f'  max|dp_out|  = {max_dp:.6e}')

In [ ]:
# Concrete illustration for one perturbation
example_seq = validation_sequences['true_branch']
row_index = 3   # row t3
constant = 2.5

base = forward_from_R(example_seq, R6)
R_mod = add_constant_to_row(R6, row_index, constant)
mod = forward_from_R(example_seq, R_mod)

print(f'Example sequence: {example_seq}')
print(f'Perturbation: add {constant:+.1f} to every entry of row {row_index} ({dim_labels[row_index]}) of R')
print('\nOriginal row of R:')
print(R6[row_index].detach().numpy())
print('\nModified row of R:')
print(R_mod[row_index].detach().numpy())
print('\nOriginal S:')
print(base['S'].detach().numpy())
print('\nModified S:')
print(mod['S'].detach().numpy())
print('\nOriginal P:')
print(base['P'].detach().numpy())
print('\nModified P:')
print(mod['P'].detach().numpy())
print('\nOriginal p_out:')
print(base['p_out'].detach().numpy())
print('\nModified p_out:')
print(mod['p_out'].detach().numpy())

In [ ]:
# Compact invariance verdict
max_dS = max(item['max_abs_diff_S'] for item in results)
max_dP = max(item['max_abs_diff_P'] for item in results)
max_dH = max(item['max_abs_diff_H'] for item in results)
max_dz = max(item['max_abs_diff_z'] for item in results)
max_dp = max(item['max_abs_diff_p_out'] for item in results)

print('Compact invariance verdict:')
print(f'  max|dS|     = {max_dS:.6f}')
print(f'  max|dP|     = {max_dP:.6e}')
print(f'  max|dH|     = {max_dH:.6e}')
print(f'  max|dz|     = {max_dz:.6e}')
print(f'  max|dp_out| = {max_dp:.6e}')

if max_dP < 1e-6 and max_dp < 1e-6:
    print('\nConclusion: row-additive perturbations change S but leave attention probabilities and final outputs unchanged to numerical precision.')
else:
    print('\nConclusion: invariance failed numerically.')

## Conclusion

For Example 6, the relationship matrix is

$$
R = e_{p3} w_K^T,
$$

so it has exactly one nonzero row, namely the $p_3$ row. Numerically that row is

$$
[0, 4, 0, -4, 0, 4, -6, -1, -6].
$$

If we add a constant to any row of $R$, then:

- $R$ changes,
- $S = X R X^T$ changes,
- but each affected score row is shifted by a constant across all columns,
- so row-wise softmax is unchanged,
- and therefore the attention probabilities and final predictions are unchanged.

So the experiment is **not invariant as a raw matrix identity**, but it **is invariant at the level of attention behavior and output**.